# Aura Graph Analytics using Models

<a target="_blank" href="https://colab.research.google.com/github/neo4j/graph-data-science-client/blob/main/examples/graph-analytics-serverless.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This Jupyter notebook is hosted [here](https://github.com/neo4j/graph-data-science-client/blob/main/examples/graph-analytics-serverless.ipynb) in the Neo4j Graph Data Science Client Github repository.

The notebook shows how to use the `graphdatascience` Python library to create, manage, and use a GDS Session.

We consider a graph of people and fruits, which we're using as a simple example to show how to connect your AuraDB instance to a GDS Session, run algorithms, and eventually write back your analytical results to the AuraDB database. 
We will cover all management operations: creation, listing, and deletion.

If you are using self managed DB, follow [this example](../graph-analytics-serverless-self-managed).

## Prerequisites

This notebook requires having an AuraDB instance available and have the Aura Graph Analytics [feature](https://neo4j.com/docs/aura/graph-analytics/#aura-gds-serverless) enabled for your project.

You also need to have the `graphdatascience` Python library installed, version `1.15` or later.

In [ ]:
%pip install "graphdatascience>=1.20" python-dotenv "neo4j_viz[gds]"

In [ ]:
from dotenv import load_dotenv

# This allows to load required secrets from `.env` file in local directory
# This can include Aura API Credentials and Database Credentials.
# If file does not exist this is a noop.
load_dotenv("staging_ci_enterprise.env")

## Aura API credentials

The entry point for managing GDS Sessions is the `GdsSessions` object, which requires creating [Aura API credentials](https://neo4j.com/docs/aura/api/authentication).

In [ ]:
import os

from graphdatascience.session import AuraAPICredentials, GdsSessions

# you can also use AuraAPICredentials.from_env() to load credentials from environment variables
api_credentials = AuraAPICredentials(
    client_id=os.environ["CLIENT_ID"],
    client_secret=os.environ["CLIENT_SECRET"],
    # If your account is a member of several project, you must also specify the project ID to use
    project_id=os.environ.get("PROJECT_ID", None),
)

sessions = GdsSessions(api_credentials=api_credentials)

## Creating a new session

A new session is created by calling `sessions.get_or_create()` with the following parameters:

* A session name, which lets you reconnect to an existing session by calling `get_or_create` again.
* The `DbmsConnectionInfo` containing the address, user name and password to an AuraDB instance
* The session memory. 
* The cloud location.
* A time-to-live (TTL), which ensures that the session is automatically deleted after being unused for the set time, to avoid incurring costs.

See the API reference [documentation](https://neo4j.com/docs/graph-data-science-client/current/api/sessions/gds_sessions/#graphdatascience.session.gds_sessions.GdsSessions.get_or_create) or the manual for more details on the parameters.

In [ ]:
from graphdatascience.session import AlgorithmCategory, SessionMemory

# Explicitly define memory
memory = SessionMemory.m_2GB

# Estimate the memory needed for the GDS session
memory = sessions.estimate(
    node_count=20,
    relationship_count=50,
    algorithm_categories=[AlgorithmCategory.NODE_EMBEDDING],
)

print(f"Estimated memory for the session: {memory}")

In [ ]:
from datetime import timedelta

from graphdatascience.session import CloudLocation

# Create a GDS session!
gds = sessions.get_or_create(
    # we give it a representative name
    session_name="training_session",
    memory=memory,
    db_connection=None,
    ttl=timedelta(minutes=30),
    cloud_location=CloudLocation("gcp", "europe-west1"),
)

In [ ]:
# Verify the connectivity. Hints towards TLS or firewall issues if this fails directly after get_or_create
gds.verify_connectivity()

## Listing sessions

You can use `sessions.list()` to see the details for each created session.

In [ ]:
from pandas import DataFrame

gds_sessions = sessions.list()

# for better visualization
DataFrame(gds_sessions)

## Projecting Graphs

Now that we have imported a graph to our database, we can project it into our GDS Session.
We do that by using the `gds.graph.project()` endpoint.

The remote projection query that we are using selects all `Person` nodes and their `LIKES` relationships, and all `Fruit` nodes and their `LIKES` relationships.
Additionally, we project node properties for illustrative purposes.
We can use these node properties as input to algorithms, although we do not do that in this notebook.

In [ ]:
G = gds.v2.graph.datasets.load_cora()
str(G)

In [ ]:
# Let us visualize the projected graph
from neo4j_viz.gds import from_gds

VG = from_gds(gds, gds.graph.get(G.name()), max_node_count=50)
for node in VG.nodes:
    node.caption = node.properties.get("name")

VG.render(initial_zoom=1.2)

## Training the GraphSage model

You can run algorithms on the constructed graph using the standard GDS Python Client API. See the other tutorials for more examples.

In [ ]:
print("Running GraphSage ...")
model, result = gds.v2.graph_sage.train(
    G, model_name="gs_example_model", feature_properties=["subject", "features"], store_model_to_disk=True
)
print(f"Training result: {result}")

In [ ]:
model, result = gds.v2.graph_sage.train(G, model_name="gs_example_model_2", feature_properties=["subject", "features"])

In [ ]:
model_name = "gs_example_model_2"

In [ ]:
store_result = gds.v2.model.store(model_name)
print(f"Model stored with result: {store_result}")

In [ ]:
model_details = gds.v2.model.get(model_name)
print(model_details)

In [ ]:
gds.v2.model.delete(model_name)  # remove the persisted model. its still available in the session

In [ ]:
gds.v2.model.store(model_name)  # store it again to make sure it is available after deletion

In [ ]:
sessions.delete(session_name="training_session")  # delete the session as we are done with the training

## Create embeddings

Now we can use the model in new sessions to create embeddings for the same graph or also new graphs with the same schema.

In [ ]:
from datetime import timedelta

from graphdatascience.session import CloudLocation, SessionMemory

memory = SessionMemory.m_2GB
gds = sessions.get_or_create(
    session_name="inference_session",
    memory=memory,
    ttl=timedelta(minutes=30),
    cloud_location=CloudLocation("gcp", "europe-west1"),
)

In [ ]:
G = gds.v2.graph.datasets.load_cora()

In [ ]:
gds.v2.model.list()  # check the model is available

In [ ]:
model_details = gds.v2.model.get(model_name)
print(model_details)

In [ ]:
gds.v2.model.load(model_name)
gds.v2.graph_sage.stream(G, model_name=model_name)

In [ ]:
sessions.delete(session_name="inference_session")  # delete the session as we are done with the inference

In [ ]:
# let's also make sure the deleted session is truly gone:
sessions.list()